# Clasificador BiLSTM desde cero sobre IMDb

Implementamos una **LSTM bidireccional (BiLSTM)** paso a paso, sin usar `nn.LSTM`. Se procesan las secuencias en ambos sentidos, se concatenan los estados finales y se clasifica.

In [ ]:
# Instalación de datasets (si no está instalado)
!pip install datasets -q

In [ ]:
# Importaciones
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from collections import Counter
from tqdm import tqdm

In [ ]:
# Descarga del dataset y vocabulario
dataset = load_dataset("imdb")
train_raw = dataset["train"].shuffle(seed=42).select(range(5000))
test_raw = dataset["test"].shuffle(seed=42).select(range(1000))

def simple_tokenizer(text):
    text = text.lower()
    for ch in ['.', ',', '!', '?', '"', "'", '(', ')', '[', ']', '{', '}', ':', ';', '-', '/', '<', '>']:
        text = text.replace(ch, ' ')
    return text.split()

counter = Counter()
for example in train_raw:
    tokens = simple_tokenizer(example["text"])
    counter.update(tokens)

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
word2idx = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for word, freq in counter.most_common(9998):
    word2idx[word] = len(word2idx)

vocab_size = len(word2idx)
print(f"Tamaño del vocabulario: {vocab_size}")

In [ ]:
# Dataset y DataLoader
class IMDBDataset(Dataset):
    def __init__(self, data, word2idx):
        self.data = data
        self.word2idx = word2idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]["text"]
        label = self.data[idx]["label"]
        tokens = simple_tokenizer(text)
        ids = [self.word2idx.get(tok, self.word2idx[UNK_TOKEN]) for tok in tokens]
        return torch.tensor(ids, dtype=torch.long), torch.tensor(label, dtype=torch.long)

def collate_fn(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in sequences])
    padded = pad_sequence(sequences, batch_first=True, padding_value=0)
    return padded, torch.stack(labels), lengths

train_dataset = IMDBDataset(train_raw, word2idx)
test_dataset = IMDBDataset(test_raw, word2idx)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

In [ ]:
# Celda LSTM
class LSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        # Puertas i, f, g, o en una sola matriz
        self.W = nn.Linear(input_dim + hidden_dim, 4 * hidden_dim, bias=True)

    def forward(self, x, state):
        # x: [batch, input_dim]
        # state: (h_prev, c_prev) cada uno [batch, hidden_dim]
        h_prev, c_prev = state
        combined = torch.cat([x, h_prev], dim=1)              # [batch, input_dim + hidden_dim]
        gates = self.W(combined)                              # [batch, 4*hidden_dim]
        i, f, g, o = gates.chunk(4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)
        c = f * c_prev + i * g
        h = o * torch.tanh(c)
        return h, (h, c)

In [ ]:
# Clasificador BiLSTM manual
class BiLSTMClassifier(nn.Module):
    """
    BiLSTM que procesa la secuencia hacia adelante y hacia atrás
    con dos celdas LSTM independientes. Concatena los últimos estados ocultos.
    """
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_classes, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.hidden_dim = hidden_dim

        # Celda hacia adelante
        self.lstm_forward = LSTMCell(emb_dim, hidden_dim)
        # Celda hacia atrás
        self.lstm_backward = LSTMCell(emb_dim, hidden_dim)

        # Capa de clasificación: entrada de 2 * hidden_dim (concatenación de ambas direcciones)
        self.fc = nn.Linear(2 * hidden_dim, num_classes)

    def forward(self, x):
        # x: [batch, seq_len]
        batch, seq_len = x.shape
        emb = self.embedding(x)                              # [batch, seq_len, emb_dim]

        # -------- Hacia adelante (t = 0 .. seq_len-1) --------
        h_f = emb.new_zeros(batch, self.hidden_dim)
        c_f = emb.new_zeros(batch, self.hidden_dim)
        for t in range(seq_len):
            h_f, (h_f, c_f) = self.lstm_forward(emb[:, t, :], (h_f, c_f))
        # h_f es el último estado hacia adelante

        # -------- Hacia atrás (t = seq_len-1 .. 0) --------
        h_b = emb.new_zeros(batch, self.hidden_dim)
        c_b = emb.new_zeros(batch, self.hidden_dim)
        for t in reversed(range(seq_len)):
            h_b, (h_b, c_b) = self.lstm_backward(emb[:, t, :], (h_b, c_b))
        # h_b es el último estado hacia atrás (que corresponde al primer token)

        # Concatenar ambos estados finales
        combined = torch.cat([h_f, h_b], dim=1)             # [batch, 2*hidden_dim]
        return self.fc(combined)

In [ ]:
# Funciones de entrenamiento y evaluación
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
emb_dim = 128
hidden_dim = 128
num_classes = 2
pad_idx = 0

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for x, y, lengths in tqdm(loader, desc="Entrenando", leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=-1)
        correct += (preds == y).sum().item()
        total += x.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    for x, y, lengths in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=-1)
        correct += (preds == y).sum().item()
        total += x.size(0)
    return total_loss / total, correct / total

def train_model(model, train_loader, test_loader, epochs=10, lr=1e-3):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for epoch in range(1, epochs+1):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc = eval_epoch(model, test_loader, criterion)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(f"Epoch {epoch:2d}: train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")
    return history

In [ ]:
# Entrenamiento del modelo BiLSTM
print("Entrenando BiLSTM (implementación manual)")
bilstm_model = BiLSTMClassifier(vocab_size, emb_dim, hidden_dim, num_classes, pad_idx)
hist_bilstm = train_model(bilstm_model, train_loader, test_loader, epochs=10)

In [ ]:
# Gráficas de evolución del entrenamiento
n_epochs = len(hist_bilstm["val_loss"])
if n_epochs == 0:
    print("No hay datos para graficar. Ejecuta primero el entrenamiento.")
else:
    epochs = list(range(1, n_epochs + 1))
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, hist_bilstm["train_loss"], marker='o', label="Train loss")
    plt.plot(epochs, hist_bilstm["val_loss"], marker='s', label="Val loss")
    plt.xlabel("Época")
    plt.ylabel("Pérdida")
    plt.title("Pérdida durante el entrenamiento")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, hist_bilstm["train_acc"], marker='o', label="Train accuracy")
    plt.plot(epochs, hist_bilstm["val_acc"], marker='s', label="Val accuracy")
    plt.xlabel("Época")
    plt.ylabel("Exactitud")
    plt.title("Exactitud durante el entrenamiento")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# Inferencia de ejemplo
def predict_sentiment(model, sentence, word2idx):
    model.eval()
    tokens = simple_tokenizer(sentence)
    ids = [word2idx.get(tok, 1) for tok in tokens]
    x = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)
        pred = logits.argmax(dim=-1).item()
    return "positivo" if pred == 1 else "negativo"

print("Prueba de sentimiento con la BiLSTM:")
print(predict_sentiment(bilstm_model, "This movie was fantastic, I loved it", word2idx))
print(predict_sentiment(bilstm_model, "Terrible film, waste of time", word2idx))

## ¿Cómo funciona la BiLSTM manual?

1. **Dos celdas LSTM independientes** (`lstm_forward`, `lstm_backward`).
2. **Hacia adelante**: recorremos `t=0..seq_len-1`, actualizando el estado `(h_f, c_f)`.
3. **Hacia atrás**: recorremos en orden inverso `t=seq_len-1..0`, con su propio estado `(h_b, c_b)`.
4. **Concatenación**: unimos los últimos estados ocultos de ambas direcciones (`h_f` y `h_b`) → vector de tamaño `2 * hidden_dim`.
5. **Clasificador**: capa `Linear` que mapea al número de clases.

Todo el proceso se desenrolla manualmente, pero el grafo dinámico de PyTorch permite aplicar automáticamente **Backpropagation Through Time (BPTT)** en ambas direcciones al llamar `loss.backward()`.

## Comparación con Elman / Jordan / LSTM unidireccional
Puedes reentrenar los modelos anteriores con los mismos hiperparámetros y comparar las curvas de exactitud. La BiLSTM suele obtener mejor rendimiento al capturar contexto tanto pasado como futuro (muy útil en análisis de sentimiento).